In [304]:
# import all Imp Libraries
import pandas as pd
import numpy as np
import re
import string
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer


In [305]:
# Puct , Stopwords and wordnet
nltk.download('stopwords')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))

# Keep important negation words
negation_words = {
    'no', 'not', 'nor', 'never',
    'neither', 'hardly', 'scarcely', 'barely'
}

stop_words = stop_words - negation_words
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [306]:
# load the dataset
df=pd.read_csv('/content/sentiment_dataset_raw.csv')
df

,Review,sentiment
0,I absolutely loved this movie! 😊 😍,positive
1,"Terrible experience, would not recommend. !!",negative
2,I absolutely loved this movie! 😊 😍,positive
3,"Had a great time with friends, everything was ...",positive
4,I absolutely loved this movie! 😊,positive
...,...,...
5995,What a fantastic experience. Will come again! !!!,positive
5996,Worst product ever. Total waste of money!,negative
5997,"Had a great time with friends, everything was ...",positive
5998,"The service was excellent, highly recommend!",positive


In [307]:
# Lowercase : convert all capital words into lowecase form
df['Review']=df['Review'].str.lower()
df

,Review,sentiment
0,i absolutely loved this movie! 😊 😍,positive
1,"terrible experience, would not recommend. !!",negative
2,i absolutely loved this movie! 😊 😍,positive
3,"had a great time with friends, everything was ...",positive
4,i absolutely loved this movie! 😊,positive
...,...,...
5995,what a fantastic experience. will come again! !!!,positive
5996,worst product ever. total waste of money!,negative
5997,"had a great time with friends, everything was ...",positive
5998,"the service was excellent, highly recommend!",positive


In [308]:
# remove HTML Tags :
def remove_html_tags(text):
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text()

df['Review']=df['Review'].apply(remove_html_tags)
df

,Review,sentiment
0,i absolutely loved this movie! 😊 😍,positive
1,"terrible experience, would not recommend. !!",negative
2,i absolutely loved this movie! 😊 😍,positive
3,"had a great time with friends, everything was ...",positive
4,i absolutely loved this movie! 😊,positive
...,...,...
5995,what a fantastic experience. will come again! !!!,positive
5996,worst product ever. total waste of money!,negative
5997,"had a great time with friends, everything was ...",positive
5998,"the service was excellent, highly recommend!",positive


In [309]:
# Remove Punctuation marks
def remove_punctuation(text):
    translator = str.maketrans("", "", string.punctuation)
    return text.translate(translator)

df['Review']=df['Review'].apply(remove_punctuation)
df

,Review,sentiment
0,i absolutely loved this movie 😊 😍,positive
1,terrible experience would not recommend,negative
2,i absolutely loved this movie 😊 😍,positive
3,had a great time with friends everything was p...,positive
4,i absolutely loved this movie 😊,positive
...,...,...
5995,what a fantastic experience will come again,positive
5996,worst product ever total waste of money,negative
5997,had a great time with friends everything was p...,positive
5998,the service was excellent highly recommend,positive


In [310]:
# chat word treamtment
# chat word treatment for feedback & review domain
chat_words_map_dict = {
    "ASAP": "As Soon As Possible",
    "CS": "Customer Support",
    "Cx": "Customer Experience",
    "DOA": "Dead On Arrival",
    "ETA": "Estimated Time Of Arrival",
    "FAQ": "Frequently Asked Questions",
    "FWIW": "For What Its Worth",
    "FYI": "For Your Information",
    "IDK": "I Do Not Know",
    "IMHO": "In My Humble Opinion",
    "IMO": "In My Opinion",
    "IRL": "In Real Life",
    "NGL": "Not Gonna Lie",
    "OOTB": "Out Of The Box",
    "PLS": "Please",
    "PLZ": "Please",
    "POV": "Point Of View",
    "QA": "Quality Assurance",
    "QC": "Quality Control",
    "RMA": "Return Merchandise Authorization",
    "ROI": "Return On Investment",
    "TBH": "To Be Honest",
    "THx": "Thanks",
    "TNx": "Thanks",
    "TIA": "Thanks In Advance",
    "TLDR": "Too Long Did Not Read",
    "UI": "User Interface",
    "Ux": "User Experience",
    "WRT": "With Respect To",
    "YMMV": "Your Mileage May Vary"
}

In [311]:
def expand_chat_words(text):
    return ' '.join([chat_words_map_dict[word] if word in chat_words_map_dict else word for word in text.split()])

df['Review'] = df['Review'].apply(expand_chat_words)
df

,Review,sentiment
0,i absolutely loved this movie 😊 😍,positive
1,terrible experience would not recommend,negative
2,i absolutely loved this movie 😊 😍,positive
3,had a great time with friends everything was p...,positive
4,i absolutely loved this movie 😊,positive
...,...,...
5995,what a fantastic experience will come again,positive
5996,worst product ever total waste of money,negative
5997,had a great time with friends everything was p...,positive
5998,the service was excellent highly recommend,positive


In [312]:
# spelling correction
# use tools ---> TextBlob , SymSpell or autocorrect


In [313]:
from textblob import TextBlob

# def correct_spelling(text):
#     return str(TextBlob(text).correct())

# df['Review'] = df['Review'].apply(correct_spelling)
# df

In [314]:
# Remove stopwords
def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word not in stop_words])

df['Review'] = df['Review'].apply(remove_stopwords)
df

,Review,sentiment
0,absolutely loved movie 😊 😍,positive
1,terrible experience would not recommend,negative
2,absolutely loved movie 😊 😍,positive
3,great time friends everything perfect lol,positive
4,absolutely loved movie 😊,positive
...,...,...
5995,fantastic experience come,positive
5996,worst product ever total waste money,negative
5997,great time friends everything perfect,positive
5998,service excellent highly recommend,positive


In [315]:
# Handling Emojis
# emojis ----> convert into the words
!pip install emoji
import emoji
df['Review']=df['Review'].apply(lambda x: emoji.demojize(x))
df

,Review,sentiment
0,absolutely loved movie :smiling_face_with_smil...,positive
1,terrible experience would not recommend,negative
2,absolutely loved movie :smiling_face_with_smil...,positive
3,great time friends everything perfect lol,positive
4,absolutely loved movie :smiling_face_with_smil...,positive
...,...,...
5995,fantastic experience come,positive
5996,worst product ever total waste money,negative
5997,great time friends everything perfect,positive
5998,service excellent highly recommend,positive


In [316]:
# prefix and suffix ----> remove
# maintaine the proper word ==> root word
# steming
stemmer = PorterStemmer()

def stem_words(text):
    return ' '.join([stemmer.stem(word) for word in text.split()])

df['Review'] = df['Review'].apply(stem_words)
df

,Review,sentiment
0,absolut love movi :smiling_face_with_smiling_e...,positive
1,terribl experi would not recommend,negative
2,absolut love movi :smiling_face_with_smiling_e...,positive
3,great time friend everyth perfect lol,positive
4,absolut love movi :smiling_face_with_smiling_e...,positive
...,...,...
5995,fantast experi come,positive
5996,worst product ever total wast money,negative
5997,great time friend everyth perfect,positive
5998,servic excel highli recommend,positive


In [317]:
# Lemmatization
# convert words to dict form ('Better' --> 'good', 'Nice'--> good)
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    # 1. Split text into words (NOT characters)
    words = text.split()

    # 2. Lemmatize each word
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]

    # 3. Join back with spaces
    return ' '.join(lemmatized_words)

# Apply to your DataFrame
df['Review_Sent'] = df['Review'].apply(lemmatize_text)
df

,Review,sentiment,Review_Sent
0,absolut love movi :smiling_face_with_smiling_e...,positive,absolut love movi :smiling_face_with_smiling_e...
1,terribl experi would not recommend,negative,terribl experi would not recommend
2,absolut love movi :smiling_face_with_smiling_e...,positive,absolut love movi :smiling_face_with_smiling_e...
3,great time friend everyth perfect lol,positive,great time friend everyth perfect lol
4,absolut love movi :smiling_face_with_smiling_e...,positive,absolut love movi :smiling_face_with_smiling_e...
...,...,...,...
5995,fantast experi come,positive,fantast experi come
5996,worst product ever total wast money,negative,worst product ever total wast money
5997,great time friend everyth perfect,positive,great time friend everyth perfect
5998,servic excel highli recommend,positive,servic excel highli recommend


In [318]:
# vectorization /feature_extraction(It is used for convert Text_data ---> Numerical_data)
### Bagofwords
from sklearn.feature_extraction.text import CountVectorizer

In [319]:
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer

lemmatizer = WordNetLemmatizer()

# 1. Clean Lemmatization Function (word by word)
# def lemmatize_text(text):
#     words = text.split()  # splits into words, NOT characters
#     return ' '.join([lemmatizer.lemmatize(w) for w in words])

# 2. Apply to raw text reviews
df['Review_Sent'] = df['Review'].apply(lemmatize_text)

# 3. Create Bag of Words
vectorizer = CountVectorizer()
bow = vectorizer.fit_transform(df['Review_Sent'])

bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 28598 stored elements and shape (6000, 40)>

In [320]:
# ngram
from sklearn.feature_extraction.text import CountVectorizer
vectorizer=CountVectorizer(ngram_range=(1,2))
ngram=vectorizer.fit_transform(df['Review_Sent'])
ngram

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 51196 stored elements and shape (6000, 100)>

In [321]:
# TFIDF
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(ngram_range=(1, 2),
    min_df=2,
    max_df=0.95)
tfidf=vectorizer.fit_transform(df['Review_Sent'])
x_tfidf=tfidf

In [322]:
x_tfidf.shape

(6000, 100)

In [323]:
x=x_tfidf
features_list=[x[i].toarray().flatten() for i in range((x.shape[0]))]
sparse_list=[x[i] for i in range((x.shape[0]))]
sparse_list

[<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 11 stored elements and shape (1, 100)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 9 stored elements and shape (1, 100)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 11 stored elements and shape (1, 100)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 11 stored elements and shape (1, 100)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 7 stored elements and shape (1, 100)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 9 stored elements and shape (1, 100)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 9 stored elements and shape (1, 100)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 11 stored elements and shape (1, 100)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 7 stored elements and shape (1, 100)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 7 sto

In [324]:
from scipy.sparse import save_npz,load_npz
save_npz('tfidf_matrix.npz',x_tfidf)

In [325]:
x=load_npz('tfidf_matrix.npz')
y=df['sentiment']

In [326]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y=le.fit_transform(y)
y

array([1, 0, 1, ..., 1, 1, 0])

In [327]:
print(le.classes_)

['negative' 'positive']


In [328]:
x_array=x.toarray()
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x_array,y,test_size=0.2,random_state=42)

In [329]:
# Train Logistic Regression
from sklearn.linear_model import LogisticRegression
model=LogisticRegression()
model.fit(x_train,y_train)


LogisticRegression()

In [330]:
# model evaluation
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
y_pred=model.predict(x_test)
print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

1.0
[[571   0]
 [  0 629]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       571
           1       1.00      1.00      1.00       629

    accuracy                           1.00      1200
   macro avg       1.00      1.00      1.00      1200
weighted avg       1.00      1.00      1.00      1200



In [331]:
# Sample new data
new_reviews = [
    "This was an incredible movie, highly recommended!",
    "Terrible customer service and broken product."
]

# 1. Vectorize the new text using the ExISTING vectorizer (fitted on training data)
x_new = vectorizer.transform(new_reviews)

# 2. Predict the sentiment class (e.g., 0 for Negative, 1 for Positive)
predictions = model.predict(x_new)

# 3. Optional: Get confidence scores/probabilities
probabilities = model.predict_proba(x_new)

# Display results
for review, pred, prob in zip(new_reviews, predictions, probabilities):
    sentiment = "Positive" if pred == 0 else "Negative"
    confidence = max(prob) * 100
    print(f"Review: '{review}'")
    print(f"Prediction: {sentiment} ({confidence:.2f}% confidence)\n")

Review: 'This was an incredible movie, highly recommended!'
Prediction: Negative (51.41% confidence)

Review: 'Terrible customer service and broken product.'
Prediction: Negative (59.60% confidence)



In [332]:
# gradio code


In [333]:
import gradio as gr

def predict_sentiment_gradio(text_input):

    # 1. Lowercase
    processed_text = text_input.lower()

    # 2. Remove HTML Tags
    processed_text = remove_html_tags(processed_text)

    # 3. Remove Punctuation marks
    processed_text = remove_punctuation(processed_text)

    # 4. Expand chat words
    processed_text = expand_chat_words(processed_text)

    # 5. Remove stopwords
    processed_text = remove_stopwords(processed_text)

    # 6. Handle Emojis
    processed_text = emoji.demojize(processed_text)

    # 7. Stemming
    processed_text = stem_words(processed_text)

    # 8. Lemmatization
    # lemmatize_text() expects a STRING, not a list
    processed_text = lemmatize_text(processed_text)

    # 9. Vectorize the processed text
    x_new = vectorizer.transform([processed_text])

    # 10. Predict sentiment
    prediction = model.predict(x_new)[0]

    sentiment_label = le.inverse_transform([prediction])[0]

    return sentiment_label


# Create Gradio interface
iface = gr.Interface(
    fn=predict_sentiment_gradio,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Enter a review here..."
    ),
    outputs=gr.Textbox(),
    title="Sentiment Analysis App"
)

# Launch
iface.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a1d3f60a33eb41f9fb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://a1d3f60a33eb41f9fb.gradio.live
